## 执行摘要

一家物流运营团队计划开展一项随机现场试验，比较三种司机路线调度策略（静态基线、动态重新路由、AI 优化）在三个配送区域下的表现，响应变量为平均配送延迟（分钟）。PROC GLMPOWER 以一份预设单元均值的*示例*数据集为基础，求解在 80% 和 90% 效能下检测路线策略效应所需的司机总数，并绘制随着路线间变异性增大，可实现效能如何衰减。

# 使用 PROC GLMPOWER 确定司机路线试验的样本量

## 执行摘要

一家物流运营团队即将启动一项随机现场试验，比较三种司机路线调度策略 -- **静态（Static）**基线、**动态（Dynamic）**重新路由系统和 **AI 优化（AI-Optimized）**规划器 -- 分别部署在三个配送区域（东北区、中西部、西部）。响应变量为**平均配送延迟（分钟）**。在为该试验投入车队运力之前，团队必须回答：*需要多少名司机才能可靠地检测到预期的运营改善？*

本笔记本使用 **PROC GLMPOWER** 对该试验背后的一般线性模型执行前瞻性效能与样本量分析。基于一份预设单元均值的*示例*数据集，它 (1) 求解达到 80% 和 90% 效能所需的总入组人数，用于检测路线策略与配送区域的总体效应；(2) 绘制随着路线间变异性上升，可实现效能如何下降；(3) 生成一条效能曲线以支持入组人数决策。

> **关键结论：** 在延迟误差标准差为合理值 8 分钟的假设下，约 **63 名司机**可达到 80% 效能，**83 名司机**可达到 90% 效能，用于检测路线策略效应 -- 但如果延迟变异性接近 10 分钟，即使入组 90 名司机，效能也达不到 70%，这凸显了保持路线紧凑、良好监测的价值。

---
## 1. 构建示例设计

第一步对试验布局以及团队对每个路线策略 x 配送区域组合的*预设*平均延迟进行编码。我们固定随机种子并添加可忽略的抖动，使均值看起来更真实，同时保留 3x3 的均衡结构。每个单元的 `cell_n` 权重为 1，向 GLMPOWER 表明该设计是均衡的。

In [1]:
/* ================================================================
   生成预设平均延迟的示例数据集。
   每个路线策略 x 配送区域设计单元一行。
   ================================================================ */
数据 routing_trial;
   调用 streaminit(20260531);
   长度 routing_strategy $8 depot_region $9;
   数组 strat[3]  $8 _temporary_ ('Static' 'Dynamic' 'AIOpt');
   数组 region[3] $9 _temporary_ ('Northeast' 'Midwest' 'West');
   数组 smean[3]     _temporary_ (18.0 14.5 11.0);   /* 预设策略均值 */
   数组 radj[3]      _temporary_ (1.5  0.0 -1.0);    /* 区域调整量（分钟） */
   循环 i = 1 到 3;
      循环 j = 1 到 3;
         routing_strategy = strat[i];
         depot_region     = region[j];
         mean_delay_min   = smean[i] + radj[j]
                            + rand('normal', 0, 0.4);
         cell_n           = 1;
         输出;
      结束;
   结束;
   删除 i j;
运行;

过程 打印 数据=routing_trial 标签 noobs;
   变量 routing_strategy depot_region mean_delay_min cell_n;
   标签 routing_strategy = "路线策略"
         depot_region     = "配送区域"
         mean_delay_min   = "平均配送延迟（分钟）"
         cell_n           = "单元样本量";
   标题 "示例单元均值：预设配送延迟（分钟）";
运行;


                                                   示例单元均值：预设配送延迟（分钟）                                                    

        路线策略          配送区域                      平均配送延迟（分钟）            单元样本量
Static        Northeast                       19.687507651                1
Static        Midwest                        17.9871067398                1
Static        West                           16.8240210086                1
Dynamic       Northeast                      15.9537535365                1
Dynamic       Midwest                        14.4415169858                1
Dynamic       West                           12.8636389493                1
AIOpt         Northeast                      12.6143811724                1
AIOpt         Midwest                         10.893885771                1
AIOpt         West                             9.635351403                1




NOTE: DATA routing_trial


NOTE: Wrote routing_trial (9 rows, 4 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=routing_trial

NOTE: PROC PRINT completed: 9 observations printed, 4 variables


---
## 2. 全模型设计的样本量

在设计确定后，我们让 GLMPOWER 在 80% 和 90% 效能下**求解总样本量**（`NTOTAL = .`）。`MODEL` 语句指定了一个带交互项的双向布局（`routing_strategy*depot_region`），`WEIGHT cell_n` 声明了均衡的单元权重分布，`STDDEV = 8` 是假定的配送延迟均方根误差。`NFRACTIONAL` 让过程在取整之前报告精确的小数计数。

我们还预先登记了三项计划中的 `CONTRAST` 比较 -- AI 优化 vs. 静态、动态 vs. 静态，以及任意重新路由 vs. 静态 -- 用以记录该试验旨在检验的、具有运营意义的假设。

In [2]:
/* ================================================================
   求解在 80% 和 90% 效能下检测路线策略与配送区域效应
   所需的司机总数。
   ================================================================ */
过程 glmpower 数据=routing_trial;
   分类 routing_strategy depot_region;
   标签 mean_delay_min   = "平均配送延迟（分钟）"
         routing_strategy = "路线策略"
         depot_region     = "配送区域";
   模型 mean_delay_min = routing_strategy depot_region routing_strategy*depot_region;
   权重 cell_n;
   CONTRAST "AI 优化 vs. 静态"      routing_strategy -1  0  1;
   CONTRAST "动态 vs. 静态"         routing_strategy -1  1  0;
   CONTRAST "任意重新路由 vs. 静态" routing_strategy -2  1  1;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = .
      POWER  = 0.80 0.90;
   标题 "检测路线策略效应对延迟影响所需的样本量";
运行;


                                                   示例单元均值：预设配送延迟（分钟）                                                    

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  平均配送延迟（分钟）                    
Source              路线策略                          
Source              配送区域                          
Source              路线策略*配送区域                     

                  Computed N Total                  

   Alpha   Std Dev   N Total     Power  Actual Power
--------  --------  --------  --------  ------------
  0.0500    8.0000        63    0.8000        0.8035
  0.0500    8.0000        83    0.9000        0.9014





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 3. 效能对变异性与试验规模的敏感性

样本量的答案取决于假定的误差标准差，而这一数值事先很少能精确知道。这里我们反过来提问：**固定**若干候选入组总数（`NTOTAL = 45 90 180`），并在一组可能的延迟标准差（6、8、10 分钟）上**求解可实现效能**（`POWER = .`）。得到的表格是一张敏感性地图 -- 它显示了如果实际路线变异性高于预期，每个入组方案的稳健程度。

In [3]:
/* ================================================================
   敏感性网格：候选试验规模与可能的延迟标准差下的
   可实现效能。
   ================================================================ */
过程 glmpower 数据=routing_trial;
   分类 routing_strategy depot_region;
   标签 mean_delay_min   = "平均配送延迟（分钟）"
         routing_strategy = "路线策略"
         depot_region     = "配送区域";
   模型 mean_delay_min = routing_strategy depot_region;
   权重 cell_n;
   POWER
      nfractional
      STDDEV = 6.0 8.0 10.0
      ALPHA  = 0.05
      NTOTAL = 45 90 180
      POWER  = .;
   标题 "不同变异性与试验规模情形下的可实现效能";
运行;


                                                   示例单元均值：预设配送延迟（分钟）                                                    

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  平均配送延迟（分钟）                    
Source              路线策略                          
Source              配送区域                          

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    6.0000        45    0.8084
  0.0500    8.0000        45    0.5475
  0.0500   10.0000        45    0.3729
  0.0500    6.0000        90    0.9868
  0.0500    8.0000        90    0.8681
  0.0500   10.0000        90    0.6782
  0.0500    6.0000       180    1.0000
  0.0500    8.0000       180    0.9943
  0.0500   10.0000       180    0.9434





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 4. 入组决策的效能曲线

最后，我们绘制一条**效能曲线** -- 当司机总入组数以 30 为步长从 30 增至 270 时，在预期的 8 分钟标准差下，策略加区域模型的可实现效能 -- 通过在该入组网格上求解 `POWER = .` 生成曲线，以“司机总数”对“效能”的表格化序列呈现，这样我们就能读出达到常规 80% 和 90% 目标的入组人数。

In [4]:
/* ================================================================
   效能曲线：随司机总入组数从 30 到 270（步长 30）变化的
   可实现效能，标准差取预期的 8 分钟。
   ================================================================ */
过程 glmpower 数据=routing_trial;
   分类 routing_strategy depot_region;
   标签 mean_delay_min   = "平均配送延迟（分钟）"
         routing_strategy = "路线策略"
         depot_region     = "配送区域";
   模型 mean_delay_min = routing_strategy depot_region;
   权重 cell_n;
   POWER
      nfractional
      STDDEV = 8.0
      ALPHA  = 0.05
      NTOTAL = 30 60 90 120 150 180 210 240 270
      POWER  = .;
   标题 "效能曲线：可实现效能与司机总入组数的关系";
运行;


                                                   示例单元均值：预设配送延迟（分钟）                                                    

The GLMPOWER Procedure


             Fixed Scenario Elements              

Item                Value                         
------------------  ------------------------------
Dependent Variable  平均配送延迟（分钟）                    
Source              路线策略                          
Source              配送区域                          

            Computed Power            

   Alpha   Std Dev   N Total     Power
--------  --------  --------  --------
  0.0500    8.0000        30    0.3733
  0.0500    8.0000        60    0.6887
  0.0500    8.0000        90    0.8681
  0.0500    8.0000       120    0.9500
  0.0500    8.0000       150    0.9826
  0.0500    8.0000       180    0.9943
  0.0500    8.0000       210    0.9982
  0.0500    8.0000       240    0.9995
  0.0500    8.0000       270    0.9999





NOTE: PROC GLMPOWER data=routing_trial

NOTE: PROC GLMPOWER statement used.


---
## 5. 结果解读与建议

本分析为运营团队提供了一份有理有据的入组方案：

- **基线样本量（第 2 节）。** 假设残差标准差为 8 分钟，包含策略、区域及其交互项的完整双向模型在 **63 名司机时达到 80% 效能**，在 **83 名司机时达到 90% 效能**。考虑到流失率而上调，将入组目标定在约 **90 名司机**可稳妥超过 90% 的阈值。

- **敏感性至关重要（第 3 节）。** 效能对延迟变异性高度敏感。在 90 名司机的规模下，可实现效能从约 99%（标准差=6）降至约 87%（标准差=8），再降至约 68%（标准差=10）。45 名司机的试点仅在变异性较低时（标准差=6 时约 81% 效能）足够，而当标准差达到 10 时则严重效能不足（约 37%）。其现实意义在于：投资于一致、良好监测的路线以压低变异性，其价值不亚于增加司机人数。

- **效能曲线（第 4 节）。** 针对策略加区域模型（不含交互项，即用于敏感性扫描的视角）绘制的曲线显示，可实现效能从 30 名司机时的 0.37 攀升至 60 名时的 0.69、90 名时的 0.87、120 名时的 0.95，并在 180 名以上趋于平坦，超过 0.99。将曲线与目标对照，80% 效能在约 77 名司机处达到，90% 在约 99 名处达到 -- 略高于第 2 节全模型的样本量，这是因为去掉交互项后，同样的效应被分摊到更少的模型自由度上。

**建议：** 入组约 **90 名司机**（每种路线策略 30 名，在三个配送区域间均衡分布）。在预期的 8 分钟标准差下，这一方案可为完整模型带来超过 90% 的效能，即便在更保守的简化模型曲线上也能保持约 87% 的效能，并且试验规模足够小，可在一个运营季度内完成。

*说明：* GLMPOWER 分析的是预设*设计*，而非观测结果 -- 因此这些数字的可信度取决于预设的均值和标准差。一旦初步试点获得配送延迟变异性的实证估计，团队应重新评估样本量。